# 🪖 SafeGuard AI — Fine-tuning SSD300 en Google Colab

Este notebook entrena el modelo SSD300-VGG16 con **fine-tuning** en Colab (GPU) y guarda
el checkpoint en Google Drive para usarlo después en local.

### Estrategia de fine-tuning
| Componente | Pesos iniciales | Learning Rate |
|------------|----------------|---------------|
| Backbone VGG16 | **ImageNet** (preentrenado) | `LR × 0.1` (ajuste fino) |
| Cabeza SSD | Aleatoria | `LR` (aprende desde cero) |

> ⚠️ No se usan los pesos COCO del SSD completo porque tienen 91 clases ≠ 6 nuestras.
> Solo se carga el backbone VGG16 con ImageNet.

### Flujo completo
```
1. [COLAB]  Ejecutar secciones 1–2   →  verificar entorno
2. [COLAB]  Sección 3 (opcional)     →  Optuna busca lr y batch_size óptimos
3. [COLAB]  Sección 4               →  fine-tuning con los mejores hiperparámetros
4. [LOCAL]  Descargar ssd_best.pth  →  pegar en seguridad_obra/modelos/checkpoints/
5. [LOCAL]  Ejecutar main.ipynb     →  carga checkpoint, salta entrenamiento
```

### ¿Por qué Optuna en vez de valores fijos?
Los hiperparámetros `lr` y `batch_size` afectan significativamente al resultado final.
Optuna usa **búsqueda bayesiana (TPE)**: aprende de cada trial anterior para proponer
mejores valores, siendo mucho más eficiente que un grid search exhaustivo.

| Método | Trials necesarios | Estrategia |
|--------|-------------------|------------|
| Grid search | O(n×m) | prueba todo |
| Random search | O(n) | aleatorio |
| **Optuna (TPE)** | **O(n) pero guiado** | **bayesiano** |

## 1. Conectar Google Drive y montar el proyecto

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

# ─────────────────────────────────────────────────────────────
# AJUSTA ESTA RUTA a donde hayas subido la carpeta seguridad_obra
# en tu Google Drive.
# Ejemplo: si la subiste a  Mi unidad/Master/seguridad_obra
# pon:     /content/drive/MyDrive/Master/seguridad_obra
# ─────────────────────────────────────────────────────────────
PROJECT_PATH = "/content/drive/MyDrive/seguridad_obra"

os.chdir(PROJECT_PATH)
print(f"Directorio de trabajo: {os.getcwd()}")
print(f"Archivos: {os.listdir('.')}")

Directorio de trabajo: /content/drive/MyDrive/seguridad_obra
Archivos: ['requirements.txt', 'patch_hpo.py', 'main.ipynb', 'README.md', 'modelos', 'data', 'outputs', 'app', 'webcam', 'colab_train.ipynb']


In [3]:
import sys
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

## 1.5. Copiar datos a disco local de Colab ⚡

> **¿Por qué?** Google Drive se lee vía FUSE (red). Con 2605 imágenes y `num_workers=0`,
> la GPU espera bloqueada mientras el CPU lee cada imagen de Drive → >10 min/época.
>
> Copiar a `/content/` (SSD local de Colab) reduce el tiempo a **~30–60 s/época** en A100.
> La copia es una vez por sesión (~500 MB, tarda <30 s).

| Sin copia (Drive) | Con copia (local) |
|-------------------|-------------------|
| ~0.5–2 s/imagen   | ~0.01 s/imagen    |
| >10 min/época     | ~30–60 s/época    |

In [4]:
import shutil
import modelos.dataset as _ds
import modelos.config  as _cfg
from pathlib import Path

LOCAL_DATA = Path("/content/css-data")
DRIVE_DATA = _cfg.DATA_DIR          # ruta original en Drive

if not LOCAL_DATA.exists():
    print(f"Copiando dataset a disco local de Colab...")
    print(f"  Origen : {DRIVE_DATA}")
    print(f"  Destino: {LOCAL_DATA}")
    shutil.copytree(str(DRIVE_DATA), str(LOCAL_DATA))
    n_imgs = sum(1 for _ in LOCAL_DATA.rglob("*.jpg"))
    print(f"✓ Copia completada — {n_imgs} imágenes en {LOCAL_DATA}")
else:
    n_imgs = sum(1 for _ in LOCAL_DATA.rglob("*.jpg"))
    print(f"Datos ya en caché local ({n_imgs} imágenes). Reutilizando.")

# Redirigir los módulos al disco local (sin modificar config.py)
_cfg.DATA_DIR = LOCAL_DATA
_ds.DATA_DIR  = LOCAL_DATA
print(f"\nDATA_DIR → {LOCAL_DATA}  (SSD local, ~100x más rápido que Drive)")

Datos ya en caché local (2801 imágenes). Reutilizando.

DATA_DIR → /content/css-data  (SSD local, ~100x más rápido que Drive)


## 2. Instalar dependencias y verificar GPU

In [5]:
# Colab ya tiene torch/torchvision con GPU — solo necesitamos optuna y pyyaml
!pip install optuna pyyaml -q
print("Dependencias instaladas")

Dependencias instaladas


In [6]:
import torch
from modelos.train_ssd import get_gpu_info

gpu = get_gpu_info()
device = torch.device(gpu["device"])

print(f"Dispositivo : {gpu['device'].upper()}")
if gpu["gpu_name"]:
    print(f"GPU         : {gpu['gpu_name']}")
    print(f"VRAM total  : {gpu['vram_total_gb']} GB")
    print(f"CUDA        : {gpu['cuda_version']}")
else:
    print("No hay GPU disponible.")
    print("Ve a: Entorno de ejecucion -> Cambiar tipo de entorno -> T4 GPU (o A100)")

# Determinar batch sizes disponibles segun VRAM
# T4  (~15 GB): batch hasta 16
# A100 (~40 GB): batch hasta 32
# A100 80GB    : batch hasta 64
vram = gpu.get("vram_total_gb") or 0
if vram >= 60:
    HPO_BATCH_OPTIONS = [16, 32, 64]
    print(f"\nA100 80 GB detectada -> batch options: {HPO_BATCH_OPTIONS}")
elif vram >= 30:
    HPO_BATCH_OPTIONS = [8, 16, 32]
    print(f"\nA100 40 GB detectada -> batch options: {HPO_BATCH_OPTIONS}")
elif vram >= 10:
    HPO_BATCH_OPTIONS = [4, 8, 16]
    print(f"\nT4 detectada -> batch options: {HPO_BATCH_OPTIONS}")
else:
    HPO_BATCH_OPTIONS = [4, 8]
    print(f"\nGPU pequeña / CPU -> batch options: {HPO_BATCH_OPTIONS}")

Dispositivo : CUDA
GPU         : NVIDIA A100-SXM4-40GB
VRAM total  : 42.41 GB
CUDA        : 12.8

A100 40 GB detectada -> batch options: [8, 16, 32]


In [7]:
from modelos.config import DATA_DIR, CHECKPOINTS_DIR

for split in ["train", "valid", "test"]:
    n = len(list((DATA_DIR / split / "images").glob("*.jpg")))
    print(f"  {split:>6}: {n} imágenes")

print(f"\nCheckpoints en: {CHECKPOINTS_DIR}")

   train: 2605 imágenes
   valid: 114 imágenes
    test: 82 imágenes

Checkpoints en: /content/drive/MyDrive/seguridad_obra/modelos/checkpoints


---
## 3. Búsqueda de hiperparámetros con Optuna *(opcional)*

> Esta sección es **opcional**. Si la saltas, el fine-tuning usa los valores por defecto
> (`lr=0.001`, `batch_size=8`).

### ¿Qué hace Optuna aquí?
Ejecuta **N trials** en los que cada vez:
1. Propone un valor de `lr` y `batch_size` usando el algoritmo **TPE** (Tree-structured Parzen Estimator)
2. Entrena el modelo durante **`HPO_EPOCHS` épocas** (pocas, para que sea rápido)
3. Devuelve el mejor `val_loss` alcanzado en ese trial
4. Usa ese resultado para guiar la siguiente propuesta

Al final guarda los mejores valores en `best_lr` y `best_batch_size`.

### Parámetros ajustables
| Variable | Valor | Descripción |
|----------|-------|-------------|
| `HPO_TRIALS` | **12** | Combinaciones a probar — suficiente para que TPE aprenda patrones |
| `HPO_EPOCHS` | **4** | Épocas por trial — señal útil sin pasarse de tiempo (T4, ≤3.5 h GPU) |
| `HPO_TIMEOUT` | **2700** | Límite de 45 min → deja ~2.5 h para el entrenamiento final |

> **Espacio de búsqueda para fine-tuning:**  
> `lr ∈ [5e-5, 5e-3]` (más bajo que entrenamiento desde cero porque el backbone ya converge).
> `batch_size ∈ {4, 8, 16}` (limitado por VRAM de T4 ~15 GB).

In [8]:
# ─── Parametros de la busqueda ───────────────────────────────────────────────
HPO_TRIALS  = 12    # trials Optuna (TPE aprende con ~8-12)
HPO_EPOCHS  = 4     # epocas por trial (senal util, rapido en GPU)
HPO_TIMEOUT = 2700  # 45 min maximo -> deja ~2.5 h para entrenamiento final

# ─── Rango de lr dinamico (centrado en SSD_LEARNING_RATE de config.py) ───────
from modelos.config import SSD_LEARNING_RATE, SSD_HPO_LR_LOW_FACTOR, SSD_HPO_LR_HIGH_FACTOR

HPO_LR_LOW  = SSD_LEARNING_RATE * SSD_HPO_LR_LOW_FACTOR
HPO_LR_HIGH = SSD_LEARNING_RATE * SSD_HPO_LR_HIGH_FACTOR

print(f"Rango HPO lr    : [{HPO_LR_LOW:.1e}, {HPO_LR_HIGH:.1e}]")
print(f"Rango HPO batch : {HPO_BATCH_OPTIONS}  (adaptado a {gpu['gpu_name'] or 'CPU'})")

# ─────────────────────────────────────────────────────────────────────────────
import json
import optuna
from pathlib import Path
from modelos.train_ssd import train_ssd
from modelos.config import CHECKPOINTS_DIR

optuna.logging.set_verbosity(optuna.logging.WARNING)

HPO_CKPT_DIR     = CHECKPOINTS_DIR / "hpo_tmp"
HPO_RESULTS_PATH = CHECKPOINTS_DIR / "hpo_best_params.json"
HPO_CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Si ya existe resultado guardado, cargarlo ───────────────────────────────
if HPO_RESULTS_PATH.exists():
    with open(HPO_RESULTS_PATH) as f:
        _saved = json.load(f)
    best_lr         = _saved["lr"]
    best_batch_size = _saved["batch_size"]
    study           = None
    print("\nResultados HPO ya guardados:")
    print(f"  lr={best_lr:.2e} | batch_size={best_batch_size} | val_loss={_saved['best_val_loss']:.4f}")
    print("  Borra hpo_best_params.json para repetir la busqueda.")

else:
    def objective(trial: optuna.Trial) -> float:
        lr         = trial.suggest_float("lr", HPO_LR_LOW, HPO_LR_HIGH, log=True)
        batch_size = trial.suggest_categorical("batch_size", HPO_BATCH_OPTIONS)
        print(f"\nTrial {trial.number:>3} | lr={lr:.2e} | batch_size={batch_size}")
        _, history = train_ssd(
            num_epochs      = HPO_EPOCHS,
            batch_size      = batch_size,
            lr              = lr,
            checkpoints_dir = HPO_CKPT_DIR,
            save_every      = 0,
            patience        = 0,
        )
        best_val = min(history["val_loss"])
        print(f"           -> mejor val_loss={best_val:.4f}")
        return best_val

    study = optuna.create_study(
        direction  = "minimize",
        sampler    = optuna.samplers.TPESampler(seed=42),
        study_name = "ssd_ft_hpo",
    )

    print(f"\nIniciando HPO: {HPO_TRIALS} trials x {HPO_EPOCHS} epocas")
    print(f"  lr en [{HPO_LR_LOW:.1e}, {HPO_LR_HIGH:.1e}]  |  batch en {HPO_BATCH_OPTIONS}")

    study.optimize(objective, n_trials=HPO_TRIALS, timeout=HPO_TIMEOUT)

    print(f"Mejor val_loss : {study.best_value:.4f}")
    print(f"Mejor lr       : {study.best_params['lr']:.2e}")
    print(f"Mejor batch    : {study.best_params['batch_size']}")

    # Guardar resultado
    best_lr         = study.best_params["lr"]
    best_batch_size = study.best_params["batch_size"]
    with open(HPO_RESULTS_PATH, "w") as f:
        json.dump({"lr": best_lr, "batch_size": best_batch_size,
                   "best_val_loss": study.best_value,
                   "gpu": gpu["gpu_name"], "n_trials": HPO_TRIALS}, f, indent=2)
    print(f"Guardado en: {HPO_RESULTS_PATH}")

Rango HPO lr    : [1.0e-04, 1.0e-02]
Rango HPO batch : [8, 16, 32]  (adaptado a NVIDIA A100-SXM4-40GB)

Resultados HPO ya guardados:
  lr=4.62e-03 | batch_size=8 | val_loss=7.2799
  Borra hpo_best_params.json para repetir la busqueda.


In [9]:
# ─── Visualizar evolución del estudio ───────────────────────────────────────
if study is None:
    print("HPO cargado desde archivo — visualización no disponible (no hay objeto study).")
else:
    import matplotlib.pyplot as plt

    trials_df = study.trials_dataframe()
    trials_df = trials_df.sort_values("number")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # val_loss por trial
    axes[0].plot(trials_df["number"], trials_df["value"], "o-", color="steelblue")
    axes[0].axhline(study.best_value, color="green", linestyle="--", alpha=0.7,
                    label=f"Mejor: {study.best_value:.4f}")
    axes[0].set_title("Val loss por trial")
    axes[0].set_xlabel("Trial"); axes[0].set_ylabel("Val loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # lr vs val_loss
    lrs = [t.params["lr"] for t in study.trials]
    vals = [t.value for t in study.trials]
    axes[1].scatter(lrs, vals, c="steelblue", s=60, alpha=0.8)
    axes[1].scatter([study.best_params["lr"]], [study.best_value],
                    c="green", s=120, zorder=5, label="Mejor")
    axes[1].set_xscale("log")
    axes[1].set_title("lr vs val_loss")
    axes[1].set_xlabel("lr (log)"); axes[1].set_ylabel("Val loss")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    # batch_size vs val_loss (box plot)
    bs_vals = {b: [] for b in HPO_BATCH_OPTIONS}
    for t in study.trials:
        bs_vals[t.params["batch_size"]].append(t.value)
    axes[2].boxplot([bs_vals[b] for b in HPO_BATCH_OPTIONS], tick_labels=[str(b) for b in HPO_BATCH_OPTIONS])
    axes[2].set_title("batch_size vs val_loss")
    axes[2].set_xlabel("batch_size"); axes[2].set_ylabel("Val loss")
    axes[2].grid(True, alpha=0.3, axis="y")

    plt.suptitle(f"Optuna HPO — {len(study.trials)} trials", fontsize=13)
    plt.tight_layout()
    plt.show()

    # Tabla de los 5 mejores trials
    print("\nTop-5 trials:")
    top5 = trials_df.nsmallest(5, "value")[["number", "params_lr", "params_batch_size", "value"]]
    top5.columns = ["trial", "lr", "batch_size", "val_loss"]
    print(top5.to_string(index=False))

HPO cargado desde archivo — visualización no disponible (no hay objeto study).


In [10]:
# ─── Guardar mejores hiperparámetros (variable + disco) ────────────────────
import json

if study is not None:
    best_lr         = study.best_params["lr"]
    best_batch_size = study.best_params["batch_size"]

    # Guardar en disco → las siguientes ejecuciones del notebook lo leerán
    _params = {"lr": best_lr, "batch_size": best_batch_size, "best_val_loss": study.best_value}
    with open(HPO_RESULTS_PATH, "w") as f:
        json.dump(_params, f, indent=2)
    print(f"✓ Hiperparámetros guardados en disco: lr={best_lr:.2e} | batch_size={best_batch_size}")
    print(f"  → {HPO_RESULTS_PATH}")
else:
    # Ya cargados en celda anterior desde hpo_best_params.json
    print(f"Hiperparámetros ya listos (cargados desde archivo): lr={best_lr:.2e} | batch_size={best_batch_size}")


Hiperparámetros ya listos (cargados desde archivo): lr=4.62e-03 | batch_size=8


---
## 4. Fine-tuning SSD300 (100 épocas, early stopping patience=10)

Usa los hiperparámetros encontrados por Optuna (si ejecutaste la sección 3)
o los valores por defecto si la saltaste.

**Estrategia completa:**
| Componente | Detalle |
|------------|---------|
| Backbone VGG16 | Pesos ImageNet, LR × 0.1 |
| Cabeza SSD | Aleatoria, LR completo |
| Épocas máx. | 100 |
| Early stopping | Para si 10 épocas seguidas sin mejorar `val_loss` |
| Scheduler | MultiStepLR (baja LR en época 40 y 70) |

**Preprocesamiento train:** HFlip(p=0.5) + ColorJitter + GaussianBlur(p=0.2)  
**Preprocesamiento val/test:** solo ToTensor (sin augmentación para reproducibilidad)

> El checkpoint se guarda en `modelos/checkpoints/ssd_best.pth` en tu Google Drive.

In [ ]:
from modelos.train_ssd import build_ssd_model, train_ssd, load_checkpoint
from modelos.config import CHECKPOINTS_DIR, SSD_NUM_EPOCHS

# ─── Hiperparámetros: Optuna si se ejecutó, defaults si se saltó ────────────
try:
    _lr    = best_lr
    _batch = best_batch_size
    print(f"Usando hiperparámetros de Optuna: lr={_lr:.2e} | batch_size={_batch}")
except NameError:
    _lr    = 0.001
    _batch = 8
    print(f"HPO no ejecutado -> defaults fine-tuning: lr={_lr} | batch_size={_batch}")

ssd_checkpoint = CHECKPOINTS_DIR / "ssd_best.pth"
ssd_history    = None

if ssd_checkpoint.exists():
    print(f"\nCheckpoint ya existe: {ssd_checkpoint}")
    print("Borra el archivo y ejecuta de nuevo para reentrenar.")
    ssd_model = build_ssd_model().to(device)
    load_checkpoint(ssd_model, ssd_checkpoint, device=device)
    ssd_model.eval()
else:
    print(f"\nIniciando fine-tuning con GPU...")
    print(f"  lr={_lr:.2e} | batch_size={_batch} | epocas_max={SSD_NUM_EPOCHS} | patience=10")
    print(f"  Preprocesamiento: HFlip(p=0.5) + ColorJitter + GaussianBlur(p=0.2)")
    ssd_model, ssd_history = train_ssd(
        num_epochs = SSD_NUM_EPOCHS,   # 100 epocas maximo
        batch_size = _batch,
        lr         = _lr,
        patience   = 10,               # early stopping: para si 10 epocas sin mejora
    )

Usando hiperparámetros de Optuna: lr=4.62e-03 | batch_size=8

Iniciando fine-tuning con GPU...
  lr=4.62e-03 | batch_size=8 | epocas_max=40 | patience=10
  Preprocesamiento: HFlip(p=0.5) + ColorJitter + GaussianBlur(p=0.2)
Dispositivo      : cuda
GPU              : NVIDIA A100-SXM4-40GB  (42.41 GB VRAM)
Fine-tuning SSD300-VGG16  (backbone ImageNet preentrenado)
  LR cabeza      : 4.62e-03
  LR backbone    : 4.62e-04  (factor=0.1)
  Batch size     : 8
  Epocas max     : 40  |  Early stopping patience=10
  Preprocesamiento: HFlip(p=0.5) + ColorJitter + GaussianBlur(p=0.2)
Dataset — train: 2605 imgs | valid: 114 imgs
Augmentacion train: HFlip(p=0.5) + ColorJitter + GaussianBlur(p=0.2)

Parametros del modelo:
  Backbone (VGG16) :   22,943,936
  Cabeza SSD       :    1,336,620
  Total            :   24,280,556

 Epoca | Train loss |   Val loss |        LR |   Tiempo
-------------------------------------------------------
     1 |    10.8406 |     8.8031 |  4.62e-04 |   2m 14s
         -> Me

In [ ]:
# ─── Curva de pérdidas (si se acaba de entrenar) ───────────────────────────
if ssd_history is not None:
    import matplotlib.pyplot as plt

    stopped    = ssd_history["stopped_epoch"]
    epochs_rng = range(1, stopped + 1)
    best_epoch = ssd_history["val_loss"].index(min(ssd_history["val_loss"])) + 1
    best_loss  = min(ssd_history["val_loss"])

    plt.figure(figsize=(10, 4))
    plt.plot(epochs_rng, ssd_history["train_loss"], label="Train Loss")
    plt.plot(epochs_rng, ssd_history["val_loss"],   label="Val Loss")
    plt.axvline(x=best_epoch, color="green", linestyle="--", alpha=0.7,
                label=f"Mejor época ({best_epoch})")
    plt.scatter([best_epoch], [best_loss], color="green", zorder=5)
    plt.title(f"SSD300 — paró en época {stopped} | mejor val_loss={best_loss:.4f}\n"
              f"lr={_lr:.2e} | batch_size={_batch}")
    plt.xlabel("Época"); plt.ylabel("Loss")
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Modelo cargado desde checkpoint — no hay historial de entrenamiento.")

---
## 5. ¿Dónde está el checkpoint?

El archivo `ssd_best.pth` ya está guardado en tu **Google Drive** dentro de la carpeta del proyecto.
Para usarlo en local solo tienes que descargarlo y copiarlo a la carpeta correcta.


In [ ]:
# ─── Verificar y mostrar ruta del checkpoint ─────────────────────────────────
if ssd_checkpoint.exists():
    size_mb = ssd_checkpoint.stat().st_size / 1e6
    print(f"✓ Checkpoint guardado correctamente")
    print(f"  Ruta en Drive : {ssd_checkpoint}")
    print(f"  Tamaño        : {size_mb:.1f} MB")
    if ssd_history is not None:
        print(f"  Entrenado con : lr={_lr:.2e} | batch_size={_batch}")
        print(f"  Mejor val_loss: {min(ssd_history['val_loss']):.4f} (época {ssd_history['val_loss'].index(min(ssd_history['val_loss']))+1})")
    print()
    print("SIGUIENTE PASO — en tu ordenador local:")
    print("  1. Descarga el archivo desde Google Drive:")
    print(f"     {ssd_checkpoint.name}")
    print("  2. Cópialo a:")
    print("     seguridad_obra/modelos/checkpoints/ssd_best.pth")
    print("  3. Abre main.ipynb y ejecuta normalmente.")
    print("     La sección 2 cargará el checkpoint en vez de entrenar.")
else:
    print("No se encontró el checkpoint. Ejecuta la celda de entrenamiento primero.")

---
## 6. Log de entrenamiento

Metricas completas guardadas en `modelos/checkpoints/ssd_training_log.json`.
Utiles para incluir en la memoria del proyecto.

In [ ]:
import json
import matplotlib.pyplot as plt
from modelos.config import CHECKPOINTS_DIR

log_path = CHECKPOINTS_DIR / "ssd_training_log.json"

if not log_path.exists():
    print("No hay log todavia. Entrena primero en la seccion 4.")
else:
    with open(log_path) as f:
        log = json.load(f)

    t  = log["training"]
    hw = log["metadata"]
    m  = log["model"]
    hp = log["hyperparameters"]

    print("  METRICAS DE ENTRENAMIENTO — para la memoria")
    print(f"\n[Hardware]")
    print(f"  GPU              : {hw['gpu_name'] or 'CPU'}")
    print(f"  VRAM total       : {hw['vram_total_gb']} GB")
    print(f"  CUDA             : {hw['cuda_version']}")
    print(f"  PyTorch          : {hw['torch_version']}")
    print(f"  Fecha            : {hw['date']}")

    print(f"\n[Modelo]")
    print(f"  Arquitectura     : {m['architecture']}")
    print(f"  Backbone         : {m['backbone']}")
    print(f"  Clases           : {m['num_classes']}  (5 EPIs + background)")
    print(f"  Params backbone  : {m['params']['backbone']:,}")
    print(f"  Params cabeza    : {m['params']['head']:,}")
    print(f"  Params total     : {m['params']['total']:,}")
    print(f"  Checkpoint size  : {m['checkpoint_mb']} MB")

    print(f"\n[Hiperparametros]")
    print(f"  LR cabeza        : {hp['lr_head']:.2e}")
    print(f"  LR backbone      : {hp['lr_backbone']:.2e}  (x{hp['backbone_lr_factor']})")
    print(f"  Batch size       : {hp['batch_size']}")
    print(f"  Optimizador      : {hp['optimizer']}  momentum={hp['momentum']}")
    print(f"  Scheduler        : {hp['scheduler']}  milestones={hp['lr_milestones']}")
    print(f"  Epocas max       : {hp['max_epochs']}  |  patience={hp['patience']}")

    print(f"\n[Preprocesamiento]")
    for aug in log["preprocessing"]["train"]:
        print(f"  Train  : {aug}")
    for aug in log["preprocessing"]["val_test"]:
        print(f"  Val    : {aug}")
    print(f"  Norm   : {log['preprocessing']['normalization']}")

    print(f"\n[Resultado]")
    print(f"  Epocas ejecutadas : {t['epochs_run']}")
    print(f"  Mejor epoca       : {t['best_epoch']}")
    print(f"  Mejor val_loss    : {t['best_val_loss']:.6f}")
    print(f"  Early stopping    : {'Si' if t['early_stopping_triggered'] else 'No'}")
    print(f"  Tiempo total      : {t['total_time_formatted']}")
    print(f"  Tiempo/epoca      : {t['avg_epoch_time_s']:.1f} s")
    if t.get("peak_gpu_memory_gb"):
        print(f"  Pico VRAM usado   : {t['peak_gpu_memory_gb']} GB  "
              f"({t['peak_gpu_memory_gb']/hw['vram_total_gb']*100:.1f}% de {hw['vram_total_gb']} GB)")

    # ─── Curvas de perdida ────────────────────────────────────────────────────
    epochs    = [e["epoch"]      for e in log["per_epoch"]]
    train_l   = [e["train_loss"] for e in log["per_epoch"]]
    val_l     = [e["val_loss"]   for e in log["per_epoch"]]
    epoch_t   = [e["epoch_time_s"] for e in log["per_epoch"]]
    lrs       = [e["lr_head"]    for e in log["per_epoch"]]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Loss
    axes[0].plot(epochs, train_l, label="Train loss", color="steelblue")
    axes[0].plot(epochs, val_l,   label="Val loss",   color="orange")
    axes[0].axvline(t["best_epoch"], color="green", linestyle="--", alpha=0.7,
                    label=f"Mejor epoca ({t['best_epoch']})")
    axes[0].scatter([t["best_epoch"]], [t["best_val_loss"]], color="green", zorder=5)
    axes[0].set_title("Curvas de perdida")
    axes[0].set_xlabel("Epoca"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # Tiempo por epoca
    axes[1].plot(epochs, epoch_t, color="purple", marker="o", markersize=3)
    axes[1].set_title("Tiempo por epoca")
    axes[1].set_xlabel("Epoca"); axes[1].set_ylabel("Segundos")
    axes[1].grid(True, alpha=0.3)

    # Learning rate
    axes[2].plot(epochs, lrs, color="red", drawstyle="steps-post")
    axes[2].set_title("Learning rate (cabeza)")
    axes[2].set_xlabel("Epoca"); axes[2].set_ylabel("LR")
    axes[2].set_yscale("log"); axes[2].grid(True, alpha=0.3)

    plt.suptitle(
        f"SSD300-VGG16  |  {hw['gpu_name'] or 'CPU'}  |  "
        f"Epocas: {t['epochs_run']}  |  Tiempo: {t['total_time_formatted']}",
        fontsize=12
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# ─── Alternativa: descargar directamente desde Colab ─────────────────────────
# (Si no tienes la carpeta en Drive y prefieres descarga directa)
from google.colab import files
files.download(str(ssd_checkpoint))